# RDORP — reproduction notebook

**Every quantitative claim this project makes, recomputed from the master
database and from first principles.**

Nothing here is copied from the documents. Each figure is derived in this
notebook and then *asserted* against the value the documents publish, so if the
corpus moves and a document does not, this notebook fails rather than agreeing.

Run it top to bottom. It needs `database/rdorp.sqlite`, which
`python run_pipeline.py` produces.

| Part | What it reproduces |
| ---- | ------------------ |
| 1 | Provenance — which database, how many rows |
| 2 | The corpus: composition, coverage, the British skew, quality grades |
| 3 | The scoring formula, recomputed cell by cell without the project's own scorer |
| 4 | The ranking, and every inclusion scenario |
| 5 | Evidence clustering, and the tie rule that once reversed the reported leader |
| 6 | The weighting sweep — 45 combinations |
| 7 | Predictive commitment: what each hypothesis staked |
| 8 | The seven geometric and computational experiments, from first principles |
| 9 | The blind protocols: inter-rater agreement |
| 10 | What would actually change the answer |
| 11 | Assertions — every headline figure, checked |

**On what this does and does not establish.** It establishes that the numbers
follow from the recorded data by the stated rules. It establishes nothing about
whether the directions, predictions and weights encoded in that data are
*right*: two independent specifiers agreed on about half of them (Part 9). A
reproducible analysis is not a correct one.

## Part 1 — Provenance

In [1]:
import hashlib, io, itertools, json, math, os, re, sqlite3, sys, collections
from fractions import Fraction

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "database"))
DB = os.path.join(ROOT, "database", "rdorp.sqlite")
assert os.path.exists(DB), f"database not found at {DB}; run `python run_pipeline.py` first"

con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
q  = lambda s, *a: con.execute(s, a).fetchall()
q1 = lambda s, *a: con.execute(s, a).fetchone()[0]

with open(DB, "rb") as fh:
    digest = hashlib.sha256(fh.read()).hexdigest()

print(f"database : {DB}")
print(f"sha256   : {digest[:32]}...")
print()
for t in sorted(r[0] for r in q("SELECT name FROM sqlite_master WHERE type='table'")):
    if t != "sqlite_sequence":
        print(f"  {t:24} {q1(f'SELECT COUNT(*) FROM {t}'):5}")

database : C:\Users\csern\Documents\dodecahedron\database\rdorp.sqlite
sha256   : 8f682c1f023818f222cede22b25d5815...

  artifact_observations      238
  corpus_observations         37
  evidence_register           47
  evidence_sources            50
  evidence_variables          48
  experiments                 12
  hdm_scores                 448
  hpm                        566
  hpm_readings                21
  hypotheses                  14
  predictions                 11
  results                     14
  screening                  134
  screening_candidates        17
  sources                     51
  specimen_quality            60
  specimens                   60
  utility_assessments         19


## Part 2 — The corpus

The claims under test: 40 specimens, 224 sourced observations, 49 sources,
31 % of the known corpus, **50 % British against a known corpus about 20 %
British**, 10 countries, 11 fragments.

In [2]:
specimens    = q1("SELECT COUNT(*) FROM specimens")
observations = q1("SELECT COUNT(*) FROM artifact_observations")
sources      = q1("SELECT COUNT(*) FROM sources")
countries    = q1("SELECT COUNT(DISTINCT country) FROM specimens WHERE country IS NOT NULL AND country <> ''")
KNOWN_CORPUS = 134          # c 2025, PUB-0003 - the same denominator
                            # reports.py and render_docs.py use
british      = q1("SELECT COUNT(*) FROM specimens WHERE country = 'United Kingdom'")
fragments    = q1("SELECT COUNT(*) FROM specimen_quality "
                  "WHERE LOWER(COALESCE(completeness,'')) LIKE '%fragment%'")

print(f"specimens          {specimens}")
print(f"observations       {observations}")
print(f"sources            {sources}")
print(f"countries          {countries}")
print(f"fragments          {fragments}")
print(f"coverage           {100*specimens/KNOWN_CORPUS:.1f} %  of {KNOWN_CORPUS} catalogued")
print(f"British share      {100*british/specimens:.1f} %  ({british} of {specimens})")
print()
print("by country:")
for r in q("SELECT country, COUNT(*) c FROM specimens GROUP BY 1 ORDER BY c DESC, 1"):
    print(f"  {str(r['country']):18} {r['c']:3}")

specimens          60
observations       238
sources            51
countries          10
fragments          11
coverage           44.8 %  of 134 catalogued
British share      38.3 %  (23 of 60)

by country:
  United Kingdom      23
  Germany             11
  France              10
  Switzerland          6
  Netherlands          3
  Belgium              2
  Unknown              2
  Austria              1
  Hungary              1
  Serbia               1


**Source concentration.** The claim is that 40 % of all observations come from
a single source, and that the corpus-level statistics rest on `PUB-0003`, which
summarises an unpublished catalogue.

In [3]:
print("artifact observations by source (top 5):")
by_source = q("SELECT source_id, COUNT(*) c FROM artifact_observations "
              "GROUP BY 1 ORDER BY c DESC LIMIT 5")
for r in by_source:
    print(f"  {r['source_id']}  {r['c']:4}  {100*r['c']/observations:5.1f} %")
top_src = by_source[0]
top2_pct = round(100 * (by_source[0]["c"] + by_source[1]["c"]) / observations)
print(f"\n  top source {100*top_src['c']/observations:.0f} %, "
      f"top two together {top2_pct} %")

print("\ncorpus-level observations by source:")
n_corpus = q1("SELECT COUNT(*) FROM corpus_observations")
for r in q("SELECT COALESCE(source_id,'(none)') s, COUNT(*) c FROM corpus_observations GROUP BY 1 ORDER BY c DESC"):
    print(f"  {r['s']:12} {r['c']:4}  {100*r['c']/n_corpus:5.1f} %")

print("\nobservations per specimen, British vs continental:")
sql = ("SELECT CASE WHEN country='United Kingdom' THEN 'British' ELSE 'continental' END k, "
       "COUNT(*) n, "
       "ROUND(AVG((SELECT COUNT(*) FROM artifact_observations o "
       "           WHERE o.rd_id = specimens.rd_id)), 2) mean "
       "FROM specimens GROUP BY 1")
for r in q(sql):
    print(f"  {r['k']:12} n={r['n']:3}  mean observations {r['mean']}")

artifact observations by source (top 5):
  PUB-0006    90   37.8 %
  PUB-0003    37   15.5 %
  PUB-0010    20    8.4 %
  PUB-0019    18    7.6 %
  PUB-0012    11    4.6 %

  top source 38 %, top two together 53 %

corpus-level observations by source:
  PUB-0003       28   75.7 %
  PUB-0010        3    8.1 %
  (none)          3    8.1 %
  PUB-0023        2    5.4 %
  PUB-0006        1    2.7 %

observations per specimen, British vs continental:
  British      n= 23  mean observations 5.61
  continental  n= 37  mean observations 2.95


### Quality, admissibility and the mass rule

Section 2.2 and 2.3 of RDORP-012 publish these counts. They were typed by hand
until the coverage audit found all three stale: they still carried denominators
from when the corpus was 36 specimens.

In [4]:
print("provenance grades:")
for r in q("SELECT provenance_grade g, COUNT(*) c FROM specimen_quality GROUP BY 1 ORDER BY 1"):
    print(f"  {r['g']}  {r['c']:3}")
grades = {r["g"]: r["c"] for r in
          q("SELECT provenance_grade g, COUNT(*) c FROM specimen_quality GROUP BY 1")}

print()
print("admissibility, per purpose:")
admit = {}
for col, label in (("admit_mass", "Mass"), ("admit_geometry", "Geometry"),
                   ("admit_context", "Context")):
    admit[label] = q1(f"SELECT COUNT(*) FROM specimen_quality WHERE {col} = 1")
    print(f"  {label:10} {admit[label]:3} of {specimens}")

weighed = q1("SELECT COUNT(*) FROM specimens WHERE weight_g IS NOT NULL")
frag_weighed = q1("SELECT COUNT(*) FROM specimens s JOIN specimen_quality sq "
                  "ON sq.rd_id = s.rd_id WHERE s.weight_g IS NOT NULL "
                  "AND LOWER(COALESCE(sq.completeness,'')) LIKE '%fragment%'")
print()
print(f"the mass rule: {weighed} specimens carry a weight, "
      f"{frag_weighed} of them fragments")
print("  a fragment's weight is not a specimen's weight, which is why the")
print(f"  mass rule admits only {admit['Mass']}")

provenance grades:
  A    1
  B    2
  C   27
  D   26
  E    4

admissibility, per purpose:
  Mass         6 of 60
  Geometry    11 of 60
  Context     50 of 60

the mass rule: 16 specimens carry a weight, 9 of them fragments
  a fragment's weight is not a specimen's weight, which is why the
  mass rule admits only 6


## Part 3 — The scoring formula, from scratch

RDORP-010 §10 defines

$$\text{score} = \mathrm{PRED}(p) \times \mathrm{DIR}(d), \qquad
  \text{weighted} = \text{score} \times w_{\text{power}} \times w_{\text{conf}} \times w_{\text{class}}$$

Below the tables are written out independently and every cell recomputed, then
compared against `score_hdm` and against the stored `hdm_scores` table. Three
independent routes to the same number.

In [5]:
PRED = {"++": 2.0, "+": 1.0, "0": 0.0, "-": -1.0, "--": -2.0}
DIR  = {"confirmed": 1.0, "weak_confirmed": 0.5, "ambiguous": 0.0,
        "weak_absent": -0.5, "absent": -1.0}
W_POWER = {"Very High": 3.0, "High": 2.0, "Medium": 1.0}
W_CONF  = {"A": 1.0, "B": 0.9, "C": 0.75, "D": 0.5, "E": 0.25}
W_CLASS = {"Observed": 1.0, "Experimental": 0.75, "Derived": 0.5}

power = {r["ev_id"]: r["discriminatory_power"] for r in q("SELECT * FROM evidence_variables")}
obs   = {r["ev_id"]: r for r in q("SELECT * FROM corpus_observations")}
pred  = {(r["hypothesis_id"], r["ev_id"]): r["prediction"] for r in q("SELECT * FROM hpm")}
read  = {(r["hypothesis_id"], r["ev_id"]): r["direction"] for r in q("SELECT * FROM hpm_readings")}
hyps  = [r["hypothesis_id"] for r in q("SELECT hypothesis_id FROM hypotheses ORDER BY 1")]

def cell(h, ev):
    o = obs[ev]
    r = read.get((h, ev))
    p = pred.get((h, ev), "0")            # NB: an unwritten cell scores as 0 (RDORP-013 A18)
    if r is not None:
        raw = PRED[p] * DIR[r]
    elif not o["discriminating"]:
        raw = 0.0
    else:
        raw = PRED[p] * DIR[o["direction"]]
    w = W_POWER[power[ev]] * W_CONF[o["confidence"]] * W_CLASS[o["evidence_class"]]
    return raw, raw * w

mine = {(h, ev): cell(h, ev) for ev in obs for h in hyps}
print(f"recomputed {len(mine)} cells from the published formula")

import score_hdm as S
H, V, HPM, CORPUS, READINGS, CLUSTERS = S.load(con)
theirs = S.score_all(H, V, HPM, CORPUS, READINGS)

diff = [k for k in theirs if abs(theirs[k][4] - mine[k][1]) > 1e-9]
print(f"disagreements with score_hdm.py          : {len(diff)}")

stored = {(r["hypothesis_id"], r["ev_id"]): r["weighted_score"] for r in q("SELECT * FROM hdm_scores")}
drift = [k for k, v in stored.items() if k in mine and abs(v - mine[k][1]) > 1e-6]
print(f"disagreements with the stored hdm_scores : {len(drift)}")
assert not diff and not drift, "the scoring engine does not match its own specification"

recomputed 518 cells from the published formula
disagreements with score_hdm.py          : 0
disagreements with the stored hdm_scores : 0


**How many cells are actually doing work.** A cell scores nothing when the
prediction is `0`, when the observation is `ambiguous`, or when the variable is
flagged non-discriminating and no per-cell reading overrides it.

In [6]:
nonzero = sum(1 for v in mine.values() if abs(v[1]) > 1e-9)
print(f"cells total       {len(mine)}")
print(f"cells scoring     {nonzero}   ({100*nonzero/len(mine):.0f} %)")
print(f"cells at zero     {len(mine)-nonzero}")

unwritten = [(h, ev) for ev in obs for h in hyps if (h, ev) not in pred]
by_ev = collections.Counter(ev for _h, ev in unwritten)
print(f"\ncells with NO prediction written  {len(unwritten)}  (RDORP-013 A18)")
for ev, n in sorted(by_ev.items()):
    disc = "scored" if obs[ev]["discriminating"] else "non-discriminating"
    print(f"  {ev}  {n:3}/{len(hyps)} hypotheses   {disc}")

cells total       518
cells scoring     252   (49 %)
cells at zero     266

cells with NO prediction written  56  (RDORP-013 A18)
  EV044   14/14 hypotheses   scored
  EV046   14/14 hypotheses   non-discriminating
  EV047   14/14 hypotheses   non-discriminating
  EV048   14/14 hypotheses   non-discriminating


## Part 4 — The ranking, and every scenario

In [7]:
names = {r["hypothesis_id"]: r["name"] for r in q("SELECT * FROM hypotheses")}

def totals(cells, clusters=None, tie="conservative"):
    return S.totals(cells, H, clusters=clusters, tie_rule=tie) if clusters else S.totals(cells, H)

U = totals(theirs)
order = sorted(U, key=lambda h: -U[h])
print("BASELINE — all corpus observations, fully weighted\n")
for i, h in enumerate(order, 1):
    print(f"  {i:2}. {h}  {U[h]:+7.1f}   {names[h]}")

BASELINE — all corpus observations, fully weighted

   1. H012    +24.0   Spool-knitting / cord-working frame (knob-based)
   2. H014    +21.0   Wax bulla / seal former
   3. H003    +12.2   Ritual object
   4. H008    +11.4   Portable shrine component
   5. H013     +8.7   Rope-laying top (rotated, core through one aperture)
   6. H002     +3.5   Rangefinder / measuring instrument
   7. H001     +2.3   Structural connector / modular node
   8. H005     -0.2   Textile / knitting tool
   9. H006     -0.5   Astronomical instrument
  10. H004     -0.6   Candlestick / lamp support
  11. H007     -1.5   Military equipment
  12. H011     -8.8   Archery targeting / ranging aid
  13. H010     -9.2   Parasol / umbrella crown fitting
  14. H009    -34.0   Tent apex / crown fitting (mobile shelter node)


### The tables the document publishes

`database/render_docs.py` is what writes the results tables into
`docs/RDORP-012_Results_Summary.md`, between `RDORP:BEGIN` / `RDORP:END`
markers. **It is imported here rather than reimplemented**, so the notebook and
the document cannot disagree: if these tables are right, the document's are the
same tables.

`render_docs.render(..., check=True)` verifies the committed document without
writing to it, and returns the names of any blocks that have gone stale.

In [8]:
import render_docs as RD

facts = RD.Facts(sqlite3.connect(DB))
print(f"tie rule used for every clustered figure: {RD.TIE_RULE!r}\n")

for name, build in RD.BLOCKS.items():
    print(f"--- {name} " + "-" * (68 - len(name)))
    print(build(facts))
    print()

tie rule used for every clustered figure: 'conservative'

--- composition ---------------------------------------------------------
| Specimens recorded | 60 |
| Known corpus | 116 by 2016 (`PUB-0051`), 129 catalogued to 2021 (`PUB-0023`), c 134 by 2025 (`PUB-0003`) |
| Coverage | 45 % of c 134 |
| Sourced observations | 238 |
| Sources | 51, of which 36 are graded A or B |
| Evidence variables | 48 |
| Hypotheses assessed | 14 |
| Functional domains screened | 17 |
| Experiments recorded | 12 |
| Pre-registered predictions | 11 |
| Countries represented | 10 |
| Evidence variables scored | 32 of 48 |

--- quality -------------------------------------------------------------
| Provenance grade | Meaning | Count |
| ---------------- | ------- | ----- |
| A | Excavated from a stratified, dated deposit | 1 |
| B | Excavated or reported find, documented findspot, institutional custody | 2 |
| C | Findspot recorded, surface or detector find | 27 |
| D | Institutional custody, no findspot | 

In [9]:
# Is the committed document current? This is the same check the pipeline runs.
stale = RD.render(DB, RD.DOC_DEFAULT, check=True)
print("stale blocks in docs/RDORP-012_Results_Summary.md:", stale or "none")
assert not stale, (
    "the results document does not match the database. "
    "Run `python database/render_docs.py` to bring it current.")

# Band membership is a judgement, not a computation. Any score inversion it
# creates is reported so the judgement is revisited rather than left to decay.
for note in RD.band_inversions(facts):
    print("band note:", note)

stale blocks in docs/RDORP-012_Results_Summary.md: none
band note: H013 (+17.1, Partly supported) outscores H003 (+10.3, Consistent but weakly testable)
band note: H013 (+17.1, Partly supported) outscores H008 (+9.6, Consistent but weakly testable)


### Every inclusion scenario

In [10]:
scenarios = S.build_scenarios(H, V, HPM, CORPUS, READINGS,
                              sources=S.source_counts(con) if hasattr(S, "source_counts") else None,
                              clusters=CLUSTERS)
print(f"{'scenario':24} {'vars':>5}  leader and top four")
for sc in scenarios:
    top = sorted(sc.totals, key=lambda h: -sc.totals[h])[:4]
    print(f"  {sc.key:22} {len(sc.variables_used):5}  "
          + "  ".join(f"{h} {sc.totals[h]:+.1f}" for h in top))

leaders = {sorted(sc.totals, key=lambda h: -sc.totals[h])[0] for sc in scenarios}
print(f"\nleaders across scenarios: {sorted(leaders)}"
      f"  -> {'STABLE' if len(leaders) == 1 else 'NOT STABLE'}")

scenario                  vars  leader and top four
  baseline                  32  H012 +24.0  H014 +21.0  H003 +12.2  H008 +11.4
  observed_only             28  H012 +22.4  H014 +19.3  H003 +12.2  H008 +11.4
  high_confidence           32  H012 +24.0  H014 +21.0  H003 +12.2  H008 +11.4
  very_high_power           11  H014 +10.2  H012 +9.8  H002 +6.3  H006 +2.5
  clustered                 32  H012 +23.5  H014 +20.5  H013 +17.1  H003 +10.3
  clustered_favourable      32  H014 +24.1  H012 +23.5  H013 +17.1  H003 +10.3
  unweighted                32  H012 +15.5  H014 +14.0  H013 +9.0  H003 +8.5
  multi_source              25  H012 +22.4  H014 +20.1  H013 +15.6  H003 +12.2
  same_footing              28  H012 +22.7  H014 +19.2  H003 +12.2  H008 +11.4

leaders across scenarios: ['H012', 'H014']  -> NOT STABLE


## Part 5 — Clustering, and the tie rule

Several variables restate one underlying observation. A cluster therefore
contributes its single strongest cell rather than the sum of its cells.

**Where two cells in a cluster have equal magnitude and opposite sign,
"strongest" does not pick one.** The original implementation kept whichever it
saw first, which made the published leader depend on dictionary iteration
order. This part reproduces that failure and the fix (RDORP-013 A16).

In [11]:
groups = collections.defaultdict(list)
for ev, c in CLUSTERS.items():
    groups[c].append(ev)
for c, evs in sorted(groups.items()):
    print(f"  {c:22} {sorted(evs)}")

print("\nhypothesis/cluster pairs holding an opposite-sign tie:")
ties = 0
for h in hyps:
    for c, evs in sorted(groups.items()):
        vals = [(ev, theirs[(h, ev)][4]) for ev in sorted(evs) if (h, ev) in theirs]
        if not vals:
            continue
        top = max(abs(w) for _e, w in vals)
        tied = [(e, w) for e, w in vals if abs(w) == top and top > 0]
        if len({w > 0 for _e, w in tied}) > 1:
            ties += 1
            print(f"  {h} {c:20} " + ", ".join(f"{e} {w:+.2f}" for e, w in tied))
print(f"\n{ties} such pairs — the tie rule decides each of them")

  aperture_metrics       ['EV004', 'EV005']
  casting                ['EV012', 'EV013']
  corpus_size_range      ['EV001', 'EV037', 'EV039']
  engineering_derived    ['EV033', 'EV034', 'EV035', 'EV036']
  wear                   ['EV017', 'EV018', 'EV019', 'EV020']

hypothesis/cluster pairs holding an opposite-sign tie:
  H002 engineering_derived  EV033 +1.12, EV034 -1.12
  H002 wear                 EV017 -2.25, EV019 +2.25, EV020 +2.25
  H004 engineering_derived  EV033 +1.12, EV034 -1.12
  H005 corpus_size_range    EV001 +1.80, EV039 -1.80
  H011 wear                 EV017 +2.25, EV019 -2.25
  H014 corpus_size_range    EV001 +1.80, EV039 -1.80

6 such pairs — the tie rule decides each of them


In [12]:
print("the defect, reproduced: keep the first cell seen\n")

def first_seen(cells):
    out = {h: 0.0 for h in hyps}
    best = {}
    for (h, ev), v in cells.items():
        c = CLUSTERS.get(ev)
        if c is None:
            out[h] += v[4]
        elif abs(v[4]) > abs(best.get((h, c), 0.0)):     # strictly greater: ties keep the first
            best[(h, c)] = v[4]
    for (h, _c), v in best.items():
        out[h] += v
    return out

fwd = first_seen(dict(theirs))
rev = first_seen(dict(reversed(list(theirs.items()))))
print(f"  forward order : H012 {fwd['H012']:+.2f}   H014 {fwd['H014']:+.2f}"
      f"   -> {max(fwd, key=fwd.get)} leads")
print(f"  reversed order: H012 {rev['H012']:+.2f}   H014 {rev['H014']:+.2f}"
      f"   -> {max(rev, key=rev.get)} leads")
assert max(fwd, key=fwd.get) != max(rev, key=rev.get), "expected the old rule to be order-dependent"
print("\n  the published leader depended on iteration order. This is the bug.\n")

print("the fix: an explicit, order-independent rule\n")
print(f"  {'rule':14} {'H012':>8} {'H014':>8}  leader")
for rule in S.TIE_RULES:
    a = S.totals(dict(theirs), H, clusters=CLUSTERS, tie_rule=rule)
    b = S.totals(dict(reversed(list(theirs.items()))), H, clusters=CLUSTERS, tie_rule=rule)
    assert all(abs(a[k] - b[k]) < 1e-9 for k in a), f"{rule} is still order-dependent"
    lead = max(a, key=a.get)
    print(f"  {rule:14} {a['H012']:+8.2f} {a['H014']:+8.2f}  {lead}")
print("\n  every rule is order-independent; H012 leads under all but 'favourable'")

the defect, reproduced: keep the first cell seen

  forward order : H012 +23.48   H014 +24.08   -> H014 leads
  reversed order: H012 +23.48   H014 +20.48   -> H012 leads

  the published leader depended on iteration order. This is the bug.

the fix: an explicit, order-independent rule

  rule               H012     H014  leader
  conservative     +23.48   +20.48  H012
  favourable       +23.48   +24.08  H014
  mean_tied        +23.48   +22.27  H012
  mean_all         +16.88   +16.28  H012

  every rule is order-independent; H012 leads under all but 'favourable'


In [13]:
C = S.totals(theirs, H, clusters=CLUSTERS, tie_rule="conservative")
print("effect of clustering, all fourteen:\n")
print(f"  {'H':6} {'unclustered':>12} {'clustered':>10} {'shift':>8}   name")
for h in sorted(hyps, key=lambda x: -(C[x] - U[x])):
    print(f"  {h:6} {U[h]:+12.1f} {C[h]:+10.1f} {C[h]-U[h]:+8.1f}   {names[h]}")

up = [h for h in hyps if C[h] - U[h] > 4]
down = [h for h in hyps if C[h] - U[h] < -4]
print(f"\n  gained more than 4 points: {up}")
print(f"  lost more than 4 points  : {down}")

effect of clustering, all fourteen:

  H       unclustered  clustered    shift   name
  H013           +8.7      +17.1     +8.4   Rope-laying top (rotated, core through one aperture)
  H005           -0.2       +4.6     +4.7   Textile / knitting tool
  H010           -9.2       -5.0     +4.2   Parasol / umbrella crown fitting
  H009          -34.0      -31.4     +2.6   Tent apex / crown fitting (mobile shelter node)
  H001           +2.3       +4.1     +1.7   Structural connector / modular node
  H006           -0.5       -1.0     -0.5   Astronomical instrument
  H014          +21.0      +20.5     -0.6   Wax bulla / seal former
  H012          +24.0      +23.5     -0.6   Spool-knitting / cord-working frame (knob-based)
  H004           -0.6       -1.4     -0.8   Candlestick / lamp support
  H007           -1.5       -2.4     -0.9   Military equipment
  H011           -8.8      -10.1     -1.3   Archery targeting / ranging aid
  H008          +11.4       +9.6     -1.8   Portable shrine c

## Part 6 — The weighting sweep

The three weight tables were chosen, not derived. Every combination of five
power schemes, three confidence schemes and three class schemes is re-scored:
45 in all, clustered and unclustered.

In [14]:
disc = S.scorable(CORPUS, READINGS, H)
sweep_u = S.weight_sweep(H, V, HPM, CORPUS, READINGS, disc)
sweep_c = S.weight_sweep(H, V, HPM, CORPUS, READINGS, disc, clusters=CLUSTERS)
for label, w in (("unclustered", sweep_u), ("clustered", sweep_c)):
    print(f"{label:12} {w['n']} combinations")
    for h, n in sorted(w["leaders"].items(), key=lambda x: -x[1]):
        print(f"    {h} leads {n}/{w['n']}  ({100*n/w['n']:.0f} %)"
              f"   margins {w['min_margin']:+.1f} to {w['max_margin']:+.1f}")

unclustered  45 combinations
    H012 leads 45/45  (100 %)   margins +0.9 to +6.5
clustered    45 combinations
    H012 leads 45/45  (100 %)   margins +0.1 to +6.5


## Part 7 — Predictive commitment

A hypothesis that predicts `0` everywhere risks nothing and still collects
points wherever it happened to mark `-` and the feature is absent. *Staked* is
the score it would have earned had every prediction been confirmed.

In [15]:
com = S.commitment(H, V, HPM, CORPUS, READINGS, sorted(disc))
print(f"  {'H':6} {'staked':>8} {'achieved':>9} {'ratio':>7} {'preds':>6} {'strong':>7}   name")
for h in sorted(hyps, key=lambda x: -com[x]["max_possible"]):
    mp = com[h]["max_possible"]
    print(f"  {h:6} {mp:8.1f} {U[h]:9.1f} {U[h]/mp if mp else 0:7.2f} "
          f"{com[h]['predictions_made']:6} {com[h]['strong_predictions']:7}   {names[h]}")
print("\n  the two highest ratios belong to the two hypotheses that staked least")

  H        staked  achieved   ratio  preds  strong   name
  H009       77.8     -34.0   -0.44     25      20   Tent apex / crown fitting (mobile shelter node)
  H001       68.2       2.3    0.03     28      12   Structural connector / modular node
  H011       68.0      -8.8   -0.13     23      14   Archery targeting / ranging aid
  H010       66.5      -9.2   -0.14     26      11   Parasol / umbrella crown fitting
  H013       53.3       8.7    0.16     23       8   Rope-laying top (rotated, core through one aperture)
  H002       50.6       3.5    0.07     20       8   Rangefinder / measuring instrument
  H014       46.8      21.0    0.45     21       7   Wax bulla / seal former
  H006       42.5      -0.5   -0.01     17       7   Astronomical instrument
  H005       39.4      -0.2   -0.00     18       3   Textile / knitting tool
  H012       35.8      24.0    0.67     19       3   Spool-knitting / cord-working frame (knob-based)
  H007       29.1      -1.5   -0.05     12       3   M

## Part 8 — The computational experiments, from first principles

These were the least reproducible part of the project: the results were
recorded as prose in the `experiments` table and the code that produced them
was never committed. Everything below is recomputed here from geometry.

### 8.1 The solid

Vertices are derived as the centroids of the faces of the dual icosahedron, and
checked against the property that caught an earlier error: **every vertex must
be equidistant in angle from exactly three face normals.** The first attempt at
`EXP-0003` used wrong coordinates and reported two distinct knob classes, which
would have made the choice of knob meaningful.

In [16]:
PHI = (1 + math.sqrt(5)) / 2

def normalise(v):
    n = math.sqrt(sum(x*x for x in v))
    return tuple(x/n for x in v)

# icosahedron vertices = dodecahedron face normals
ico = []
for s1 in (1, -1):
    for s2 in (1, -1):
        ico += [(0, s1*1, s2*PHI), (s1*1, s2*PHI, 0), (s2*PHI, 0, s1*1)]
face_normals = [normalise(v) for v in ico]
assert len(face_normals) == 12

# icosahedron faces -> dodecahedron vertices
def angle(a, b):
    d = max(-1.0, min(1.0, sum(x*y for x, y in zip(a, b))))
    return math.degrees(math.acos(d))

edge_len = min(angle(a, b) for a, b in itertools.combinations(face_normals, 2))
verts = []
for tri in itertools.combinations(range(12), 3):
    if all(abs(angle(face_normals[i], face_normals[j]) - edge_len) < 1e-6
           for i, j in itertools.combinations(tri, 2)):
        c = [sum(face_normals[i][k] for i in tri)/3 for k in range(3)]
        verts.append(normalise(c))
print(f"face normals (faces) : {len(face_normals)}")
print(f"vertices (knobs)     : {len(verts)}")
assert len(verts) == 20

# the check that caught the error
classes = set()
for v in verts:
    angs = sorted(round(angle(v, n), 6) for n in face_normals)
    near = [a for a in angs if abs(a - angs[0]) < 1e-6]
    classes.add((len(near), angs[0]))
print(f"\nvertex classes by (count, angle to nearest face normals): {classes}")
assert len(classes) == 1, "vertices are not equivalent — the coordinates are wrong"
n_near, ang = classes.pop()
print(f"every vertex sits at {ang:.2f}° from exactly {n_near} face normals")
print("=> the solid is VERTEX-TRANSITIVE: all 20 knobs are geometrically identical")
print("   EXP-0003 result (1): the choice of knob conveys no information.")

face normals (faces) : 12
vertices (knobs)     : 20

vertex classes by (count, angle to nearest face normals): {(3, 37.377368)}
every vertex sits at 37.38° from exactly 3 face normals
=> the solid is VERTEX-TRANSITIVE: all 20 knobs are geometrically identical
   EXP-0003 result (1): the choice of knob conveys no information.


### 8.2 EXP-0002 — can sunlight through an aperture index twelve dates?

In [17]:
angs = sorted({round(angle(a, b), 3) for a, b in itertools.combinations(face_normals, 2)})
print(f"distinct angles between the twelve face axes: {angs}")

ANNUAL_SWING = 2 * 23.44        # solar noon altitude, full annual travel
print(f"\nannual swing of solar noon altitude: {ANNUAL_SWING:.2f}°")
print(f"smallest inter-axis angle          : {min(angs):.3f}°  "
      f"= {min(angs)/ANNUAL_SWING:.2f} x the entire swing")
assert min(angs) > ANNUAL_SWING
print("=> at a fixed site at most ONE face axis can ever meet the noon sun.")

print("\nprojection reading — divisions resolvable = travel / patch")
SUN_DIAM = 0.53
def divisions(L, d):
    travel = L * math.tan(math.radians(ANNUAL_SWING))
    patch  = d + L * math.tan(math.radians(SUN_DIAM))
    return travel / patch
for name, L, d in [("Avenches", 46.5, 8.7), ("Jublains", 48.0, 10.5), ("Mainz 3", 40.0, 10.0)]:
    print(f"  {name:10} L={L:5.1f} mm  d={d:4.1f} mm  L/d={L/d:4.1f}  "
          f"-> {divisions(L, d):.1f} divisions")
need = 12
print(f"\ntwelve divisions need L/d >= ~{12*math.tan(math.radians(SUN_DIAM))/math.tan(math.radians(ANNUAL_SWING))*1:.1f}"
      f" (order 12.5); measured specimens give 3.9 to 5.6")

distinct angles between the twelve face axes: [63.435, 116.565, 180.0]

annual swing of solar noon altitude: 46.88°
smallest inter-axis angle          : 63.435°  = 1.35 x the entire swing
=> at a fixed site at most ONE face axis can ever meet the noon sun.

projection reading — divisions resolvable = travel / patch
  Avenches   L= 46.5 mm  d= 8.7 mm  L/d= 5.3  -> 5.4 divisions
  Jublains   L= 48.0 mm  d=10.5 mm  L/d= 4.6  -> 4.7 divisions
  Mainz 3    L= 40.0 mm  d=10.0 mm  L/d= 4.0  -> 4.1 divisions

twelve divisions need L/d >= ~0.1 (order 12.5); measured specimens give 3.9 to 5.6


### 8.3 EXP-0003 — suspension elevations

In [18]:
def elevations(support):
    "Elevation above horizontal of each of the six face-pair axes, support vertical."
    up = normalise(support)
    out = set()
    for n in face_normals:
        c = abs(sum(a*b for a, b in zip(up, n)))
        out.add(round(math.degrees(math.asin(min(1.0, c))), 2))
    return sorted(out)

edge_mids = []
for a, b in itertools.combinations(verts, 2):
    if abs(angle(a, b) - min(angle(x, y) for x, y in itertools.combinations(verts, 2))) < 1e-6:
        edge_mids.append(normalise([(x+y)/2 for x, y in zip(a, b)]))

modes = {"knob (20)": verts, "face (12)": face_normals, "edge (30)": edge_mids}
allel = set()
for label, sups in modes.items():
    per = {tuple(elevations(s)) for s in sups}
    assert len(per) == 1, f"{label} supports are not equivalent"
    e = sorted(per.pop())
    allel |= set(e)
    print(f"  {label:12} n={len(sups):3}  elevations {e}")
print(f"\ndistinct elevations across every mode of support: {sorted(allel)}  ({len(allel)})")

print("\nreachable by the noon sun (90 - lat + dec, |dec| <= 23.44):")
SITES = {"Arles": 43.7, "Jublains": 48.2, "Norton Disney": 53.1, "Corbridge": 55.0}
for site, lat in SITES.items():
    lo, hi = 90 - lat - 23.44, 90 - lat + 23.44
    reach = sorted(e for e in allel if lo <= e <= hi)
    print(f"  {site:14} lat {lat:4.1f}  noon altitude {lo:5.1f}–{hi:5.1f}°  reachable {reach}")
print("\n=> four elevations, the same four at every site; crossed twice a year -> 8 events, not 12")

  knob (20)    n= 20  elevations [10.81, 52.62]


  face (12)    n= 12  elevations [26.57, 90.0]
  edge (30)    n= 30  elevations [0.0, 31.72, 58.28]

distinct elevations across every mode of support: [0.0, 10.81, 26.57, 31.72, 52.62, 58.28, 90.0]  (7)

reachable by the noon sun (90 - lat + dec, |dec| <= 23.44):
  Arles          lat 43.7  noon altitude  22.9– 69.7°  reachable [26.57, 31.72, 52.62, 58.28]
  Jublains       lat 48.2  noon altitude  18.4– 65.2°  reachable [26.57, 31.72, 52.62, 58.28]
  Norton Disney  lat 53.1  noon altitude  13.5– 60.3°  reachable [26.57, 31.72, 52.62, 58.28]
  Corbridge      lat 55.0  noon altitude  11.6– 58.4°  reachable [26.57, 31.72, 52.62, 58.28]

=> four elevations, the same four at every site; crossed twice a year -> 8 events, not 12


### 8.4 EXP-0004 — could it level an aqueduct?

In [19]:
def sight_tolerance(d_near, d_far, L):
    return math.degrees(math.atan(abs(d_far - d_near) / (2 * L)))

print("horizontal sight available? edge suspension puts one face-pair axis at",
      f"{min(elevations(edge_mids[0])):.2f}°")
print()
GRADIENTS = {"Vitruvius minimum 1:200": 1/200, "Nimes 1:3000": 1/3000,
             "Aqua Marcia 1:4000": 1/4000, "flattest Nimes 1:20000": 1/20000}
print(f"{'aperture pair':28} {'tolerance':>10}")
for label, dn, df, L in [("Avenches 14.2 / 14.5 mm", 14.2, 14.5, 46.5),
                         ("typical pair, 2 mm apart", 14.0, 16.0, 46.5),
                         ("typical pair, 4.5 mm apart", 14.0, 18.5, 46.5)]:
    print(f"  {label:28} {sight_tolerance(dn, df, L):9.2f}°")
print()
best = sight_tolerance(14.2, 14.5, 46.5)
for label, g in GRADIENTS.items():
    need = math.degrees(math.atan(g))
    print(f"  {label:26} = {need:6.4f}°   best pair is {best/need:6.1f} x too coarse")

horizontal sight available? edge suspension puts one face-pair axis at 0.00°

aperture pair                 tolerance
  Avenches 14.2 / 14.5 mm           0.18°
  typical pair, 2 mm apart          1.23°
  typical pair, 4.5 mm apart        2.77°

  Vitruvius minimum 1:200    = 0.2865°   best pair is    0.6 x too coarse
  Nimes 1:3000               = 0.0191°   best pair is    9.7 x too coarse
  Aqua Marcia 1:4000         = 0.0143°   best pair is   12.9 x too coarse
  flattest Nimes 1:20000     = 0.0029°   best pair is   64.5 x too coarse


### 8.5 EXP-0005 — how many rings can fit around an aperture?

In [20]:
RATIO = math.cos(math.radians(36))
print(f"pentagon apothem / circumradius = cos 36° = {RATIO:.6f}")
print("=> no complete ring may exceed 80.9 % of the face-centre-to-knob distance,")
print("   on a dodecahedron of any size.\n")

def rings(edge_mm, aperture_mm, pitch=2.0):
    apothem = edge_mm / (2 * math.tan(math.radians(36)))   # face centre to edge midpoint
    annulus = apothem - aperture_mm / 2
    return annulus, max(0, int(annulus // pitch))

print("Vienne (RD-0035), edge 24.70 mm derived from 55 mm face-to-face:")
for ap, recorded in [(14, "4 and 6"), (22, "3"), (23, "none"), (24, "none")]:
    ann, n = rings(24.70, ap)
    print(f"  aperture {ap:4.1f} mm -> annulus {ann:5.1f} mm, room for ~{n} rings; recorded: {recorded}")
print("\n  the model predicts the decorated faces and FAILS on the undecorated pair,")
print("  which had room for 2–3 rings and carries none.\n")
print("Jublains (RD-0020), edge 21 mm — three rings on every decorated face regardless")
for ap in (9, 11, 13):
    ann, n = rings(21.0, ap)
    print(f"  aperture {ap:4.1f} mm -> annulus {ann:5.1f} mm, room for ~{n}; recorded: 3")
print("\n=> two workshop rules: Vienne holds the pitch and varies the count;")
print("   Jublains holds the count and tightens the pitch.")

pentagon apothem / circumradius = cos 36° = 0.809017
=> no complete ring may exceed 80.9 % of the face-centre-to-knob distance,
   on a dodecahedron of any size.

Vienne (RD-0035), edge 24.70 mm derived from 55 mm face-to-face:
  aperture 14.0 mm -> annulus  10.0 mm, room for ~4 rings; recorded: 4 and 6
  aperture 22.0 mm -> annulus   6.0 mm, room for ~2 rings; recorded: 3
  aperture 23.0 mm -> annulus   5.5 mm, room for ~2 rings; recorded: none
  aperture 24.0 mm -> annulus   5.0 mm, room for ~2 rings; recorded: none

  the model predicts the decorated faces and FAILS on the undecorated pair,
  which had room for 2–3 rings and carries none.

Jublains (RD-0020), edge 21 mm — three rings on every decorated face regardless
  aperture  9.0 mm -> annulus  10.0 mm, room for ~4; recorded: 3
  aperture 11.0 mm -> annulus   9.0 mm, room for ~4; recorded: 3
  aperture 13.0 mm -> annulus   8.0 mm, room for ~3; recorded: 3

=> two workshop rules: Vienne holds the pitch and varies the count;
   Ju

### 8.6 EXP-0006 — can ring counts label twelve signs?

In [21]:
OBSERVED_COUNTS = list(range(0, 7))       # 0 to 6 anywhere in the corpus
FACES = 12
print(f"distinct ring counts observed anywhere: {OBSERVED_COUNTS}  ({len(OBSERVED_COUNTS)} values)")
print(f"faces to label: {FACES}")
collisions = FACES - len(OBSERVED_COUNTS)
print(f"\npigeonhole: at least {collisions} faces must share a count with another, on every specimen")
assert collisions >= 5

vienne = {"1": 0, "1'": 0, "2": 3, "3'": 3, "5'": 3, "4'": 4, "6'": 6}
dupes = {c: [f for f, n in vienne.items() if n == c]
         for c in set(vienne.values()) if list(vienne.values()).count(c) > 1}
print(f"\nVienne, seven published faces: {vienne}")
print(f"already repeating: {dupes}")
print("=> the repetition is present before the five unpublished faces are counted.")

print("\ndoes pairing count with aperture diameter rescue it?")
vienne_ap = {"2": 22, "3'": 22, "5'": 22}
print(f"  the three faces sharing a 22 mm aperture carry {[vienne[f] for f in vienne_ap]} rings — identical")
print("  but 4' and 6' share a 14 mm aperture and carry 4 and 6, so rings are NOT wholly redundant")

distinct ring counts observed anywhere: [0, 1, 2, 3, 4, 5, 6]  (7 values)
faces to label: 12

pigeonhole: at least 5 faces must share a count with another, on every specimen

Vienne, seven published faces: {'1': 0, "1'": 0, '2': 3, "3'": 3, "5'": 3, "4'": 4, "6'": 6}
already repeating: {0: ['1', "1'"], 3: ['2', "3'", "5'"]}
=> the repetition is present before the five unpublished faces are counted.

does pairing count with aperture diameter rescue it?
  the three faces sharing a 22 mm aperture carry [3, 3, 3] rings — identical
  but 4' and 6' share a 14 mm aperture and carry 4 and 6, so rings are NOT wholly redundant


### 8.7 EXP-0007 — does the decoration determine an orientation?

In [22]:
# rotation group as permutations of the twelve faces, built by explicit construction
def rot_matrix(axis, theta):
    x, y, z = normalise(axis); c, s, t = math.cos(theta), math.sin(theta), 1-math.cos(theta)
    return ((t*x*x+c, t*x*y-s*z, t*x*z+s*y),
            (t*x*y+s*z, t*y*y+c, t*y*z-s*x),
            (t*x*z-s*y, t*y*z+s*x, t*z*z+c))

def apply(m, v):
    return tuple(sum(m[i][j]*v[j] for j in range(3)) for i in range(3))

def perm(m):
    out = []
    for n in face_normals:
        img = apply(m, n)
        j = min(range(12), key=lambda k: sum((img[a]-face_normals[k][a])**2 for a in range(3)))
        assert sum((img[a]-face_normals[j][a])**2 for a in range(3)) < 1e-9
        out.append(j)
    return tuple(out)

group = set()
axes = ([n for n in face_normals] + [v for v in verts] + edge_mids)
for ax in axes:
    for k in range(1, 6):
        try:
            group.add(perm(rot_matrix(ax, 2*math.pi*k/5)))
        except AssertionError:
            pass
        for d in (2, 3):
            try:
                group.add(perm(rot_matrix(ax, 2*math.pi*k/d)))
            except AssertionError:
                pass
group.add(tuple(range(12)))
print(f"order of the rotation group: {len(group)}")
assert len(group) == 60

pairs = {}
for i, n in enumerate(face_normals):
    j = min(range(12), key=lambda k: sum((n[a]+face_normals[k][a])**2 for a in range(3)))
    pairs[i] = j
marked = frozenset({0, pairs[0]})
stab = [g for g in group if frozenset({g[0], g[pairs[0]]}) == marked]
print(f"rotations fixing one marked opposed pair (as a set): {len(stab)}")
assert len(stab) == 10
print("\n=> marking one axis reduces 60 orientations to 10, not to 1.")
print("   Five rotations about the axis, times a flip exchanging its two ends.")
print("   If the other ten faces are identical (Jublains), all 10 survive.")
print("   Nothing observed distinguishes the two ends: there is no up and no down.")

order of the rotation group: 60
rotations fixing one marked opposed pair (as a set): 10

=> marking one axis reduces 60 orientations to 10, not to 1.
   Five rotations about the axis, times a flip exchanging its two ends.
   If the other ten faces are identical (Jublains), all 10 survive.
   Nothing observed distinguishes the two ends: there is no up and no down.


### 8.8 EXP-0009 — does the geometry fit the zodiac better than chance?

`EXP-0002` and `EXP-0003` refute the solar readings on resolution and on the
number of reachable elevations. This asks the different question that motivates
the reading in the first place: the dates the object *can* mark — do they fall
on sign boundaries?

The trap is the latitude scan. It is free, so it will beat the 7.5° chance
baseline for almost any set of elevations. The controlling comparison is
therefore a Monte Carlo over *random* elevation sets given the same freedom.

In [23]:
import exp_zodiac as Z

elev = Z.achievable_elevations()
print("face-axis elevations:", [round(e, 2) for e in elev])
# 8.3 rounds to 2 dp for display; this module keeps 6. Compare on value.
assert len(elev) == len(allel), f"{len(elev)} elevations here, {len(allel)} in 8.3"
assert all(abs(a - b) < 0.01 for a, b in zip(elev, sorted(allel))),     "the elevations must agree with those derived independently in 8.3"
print("  cross-checked against the seven derived in 8.3")

best, lat, n_events = Z.scan(elev)
print()
print(f"best mean distance to a sign boundary  {best:.2f} deg")
print(f"  at latitude                          {lat:.1f} N  over {n_events} events")
print(f"  expected of uniform dates (exact)    {Z.EXPECTED_AT_RANDOM:.2f} deg")
print(f"  optimum at the edge of the scan?     "
      f"{abs(lat - Z.LAT_MAX) < Z.LAT_STEP or abs(lat - Z.LAT_MIN) < Z.LAT_STEP}")

sims = Z.monte_carlo(len(elev), trials=5000)
p_zod = sum(1 for x in sims if x <= best) / len(sims)
median_random = sims[len(sims) // 2]
print()
print(f"{len(sims)} random elevation sets, same free scan:")
print(f"  median best fit                      {median_random:.2f} deg")
print(f"  do at least as well as the real solid {p_zod:.0%}")

# the statistic must be able to detect a real alignment, or it proves nothing
contrived = []
for lam in (30.0, 60.0, 120.0, 150.0):
    dec = math.degrees(math.asin(math.sin(math.radians(Z.OBLIQUITY))
                                 * math.sin(math.radians(lam))))
    contrived.append(90.0 - 50.0 + dec)
m_contrived, _ = Z.fit(sorted(contrived), 50.0)
print()
print(f"sensitivity check: a set placed ON boundaries scores {m_contrived:.4f} deg")
assert m_contrived < 0.01

print()
print("=> the fit is not better than chance; it is worse than the median")
print("   random solid. The apparent alignment is the free latitude scan.")

face-axis elevations: [0.0, 10.81, 26.57, 31.72, 52.62, 58.28, 90.0]
  cross-checked against the seven derived in 8.3

best mean distance to a sign boundary  4.98 deg
  at latitude                          58.0 N  over 8 events
  expected of uniform dates (exact)    7.50 deg
  optimum at the edge of the scan?     True



5000 random elevation sets, same free scan:
  median best fit                      3.51 deg
  do at least as well as the real solid 84%

sensitivity check: a set placed ON boundaries scores 0.0000 deg

=> the fit is not better than chance; it is worse than the median
   random solid. The apparent alignment is the free latitude scan.


### 8.9 EXP-0010 — could it be a volumetric measure? (C-05)

C-05 was eliminated in the screen at −18.0, argued from standardisation: a
measure that varies between examples measures nothing. That is an argument
about the corpus, not about the object. This tests the candidate on its own
terms — and the null matters, because Roman capacity units are roughly
geometric, so *any* volume in range sits within some percentage of one.

In [24]:
import exp_volume as EV

rows, mean_wall = EV.corpus_rows(DB)
print(f"volume coefficient (15+7*sqrt5)/4 = {EV.VOL_COEFF:.6f}")
print(f"mean wall where not recorded      = {mean_wall:.2f} mm")
print()
vols = []
for r in rows:
    w = r["wall_thickness_mm"] or mean_wall
    v = EV.internal_volume_ml(r["max_diameter_mm"], w)
    vols.append(v)
    unit, rel = EV.nearest_unit(v)
    print(f"  {r['specimen_name'][:26]:26} {r['max_diameter_mm']:6.1f} mm "
          f"-> {v:7.1f} ml   nearest {unit} ({rel:+.0%})")

rels = [EV.nearest_unit(v)[1] for v in vols]
observed_vol = sum(rels) / len(rels)
null = EV.null_distribution(min(vols), max(vols), n=50000)
expected_vol = sum(null) / len(null)
p_vol = sum(1 for x in null if x <= observed_vol) / len(null)
print()
print(f"mean distance to the nearest Roman unit  {observed_vol:.1%}")
print(f"same for log-uniform random volumes      {expected_vol:.1%}")
print(f"random volumes at least as close         {p_vol:.0%}")
print("=> nominal capacity is no better than chance")

print()
print("the decisive point is structural, not numerical:")
print(f"  liquid retained in any orientation      "
      f"{EV.retained_volume_ml(60.0, mean_wall):.0f} ml")
print("  every face carries an aperture at its centre, so whichever face is")
print("  down, its aperture is the lowest point of the cavity")
print(f"  cereal grain {EV.GRAIN_MM[0]:.0f}-{EV.GRAIN_MM[1]:.0f} mm against "
      f"apertures of 6-40 mm -> dry measure fails too")

volume coefficient (15+7*sqrt5)/4 = 7.663119
mean wall where not recorded      = 2.25 mm

  Couthuin/Bassenge dodecahe   44.0 mm ->    42.8 ml   nearest cyathus (+6%)
  Mainz 3 dodecahedron         55.0 mm ->    97.6 ml   nearest quartarius (+29%)
  Vienne (Isere) dodecahedro   55.0 mm ->    89.4 ml   nearest acetabulum (+31%)
  British Museum dodecahedro   56.4 mm ->    97.0 ml   nearest quartarius (+29%)
  Avenches (Aventicum) dodec   58.5 mm ->   109.2 ml   nearest quartarius (+20%)
  Jublains (Noviodunum) dode   74.0 mm ->   232.9 ml   nearest hemina (+15%)
  Norton Disney dodecahedron   80.0 mm ->   298.6 ml   nearest hemina (+9%)
  Much Hadham dodecahedron     82.0 mm ->   304.6 ml   nearest hemina (+12%)
  Louvre dodecahedron          85.0 mm ->   361.9 ml   nearest hemina (+33%)
  Fishguard dodecahedron      127.7 mm ->  1297.7 ml   nearest congius (+60%)

mean distance to the nearest Roman unit  24.3%
same for log-uniform random volumes      23.1%
random volumes at least as cl

### 8.10 EXP-0011 — do the apertures form a usable gauge series?

Five readings ask the object to *measure* by which aperture a thing passes:
shot (C-01), net mesh (C-03), dividers (C-11), garment sizes (C-13), range
(H002). All five need a graded series, agreement between examples, and
divisions a user can tell apart.

**The null is essential here.** The order statistics of a uniform sample are
evenly spaced in expectation, so *sorting alone* manufactures a progression.
Fitting a line to sorted apertures and finding a small residual proves nothing.

In [25]:
import exp_gauge as EG

ap_series = EG.measured_apertures(DB)
for nm, vals in ap_series.items():
    print(f"  {nm[:44]:44} n={len(vals):2}  {vals[0]:.1f}-{vals[-1]:.1f} mm")

print()
print("graded series? residual vs a null of sorted random sets")
graded = {}
for nm, vals in ap_series.items():
    resid = EG.linear_residual(vals)   # NB: not 'obs' - Part 3 binds that
    null = EG.null_linear_residual(len(vals), trials=5000)
    beats = 1 - sum(1 for x in null if x <= resid) / len(null)
    graded[nm] = beats
    print(f"  {nm[:40]:40} residual {resid:.4f}  median random "
          f"{null[len(null)//2]:.4f}  beats {beats:.0%}")

print()
print("divisions a user could tell apart?")
gaps = {}
for nm, vals in ap_series.items():
    gaps[nm] = EG.smallest_gap(vals)
    print(f"  {nm[:40]:40} smallest step {gaps[nm]:.2f} mm  "
          f"separable: {'yes' if gaps[nm] > EG.EYE_RESOLUTION_MM else 'NO'}")

print()
gl = EG.glans_diameters_mm()
print("lead glandes of " + ", ".join(f"{g:.0f}" for g in EG.GLANS_WEIGHTS_G)
      + " g are " + ", ".join(f"{d:.1f}" for d in gl) + " mm across")
for nm, vals in ap_series.items():
    at = sum(1 for g in gl if any(abs(g - v) < 0.5 for v in vals))
    print(f"  {nm[:40]:40} apertures AT a calibre: {at}/{len(gl)}")

n_twelve = sum(1 for v in ap_series.values() if len(v) == 12)
print()
print(f"specimens with all twelve apertures measured: {n_twelve} of {specimens}")
print("=> readability and the shot calibres are settled; regularity and")
print("   reproducibility are NOT, and only B2 can settle them.")

  RD-0020 Jublains (Noviodunum) dodecahedron   n=10  10.5-22.0 mm
  RD-0022 Mainz 3 dodecahedron                 n= 4  14.0-17.0 mm
  RD-0034 Avenches (Aventicum) dodecahedron    n=12  8.7-26.5 mm
  RD-0035 Vienne (Isere) dodecahedron          n=12  13.5-24.0 mm

graded series? residual vs a null of sorted random sets
  RD-0020 Jublains (Noviodunum) dodecahedr residual 0.0948  median random 0.0771  beats 26%


  RD-0022 Mainz 3 dodecahedron             residual 0.1394  median random 0.1160  beats 31%


  RD-0034 Avenches (Aventicum) dodecahedro residual 0.0508  median random 0.0694  beats 83%


  RD-0035 Vienne (Isere) dodecahedron      residual 0.0973  median random 0.0694  beats 15%

divisions a user could tell apart?
  RD-0020 Jublains (Noviodunum) dodecahedr smallest step 0.00 mm  separable: NO
  RD-0022 Mainz 3 dodecahedron             smallest step 0.00 mm  separable: NO
  RD-0034 Avenches (Aventicum) dodecahedro smallest step 0.30 mm  separable: NO
  RD-0035 Vienne (Isere) dodecahedron      smallest step 0.00 mm  separable: NO

lead glandes of 20, 30, 40, 50, 60 g are 15.0, 17.2, 18.9, 20.3, 21.6 mm across
  RD-0020 Jublains (Noviodunum) dodecahedr apertures AT a calibre: 2/5
  RD-0022 Mainz 3 dodecahedron             apertures AT a calibre: 2/5
  RD-0034 Avenches (Aventicum) dodecahedro apertures AT a calibre: 3/5
  RD-0035 Vienne (Isere) dodecahedron      apertures AT a calibre: 4/5

specimens with all twelve apertures measured: 2 of 60
=> readability and the shot calibres are settled; regularity and
   reproducibility are NOT, and only B2 can settle them.


### 8.11 EXP-0012 — the Wagemans sowing-calendar model (C-18)

Proposed outside this project (romandodecahedron.com): the object rests on a
face and light is sighted at solar noon through a pair of **opposed** apertures.
Each pair has its own diameters, so each gives its own limiting sun angle and so
its own date; the set is read as the window for sowing winter grain.

**The mechanism is sound** — its central constant is exactly the face-rest
elevation 8.3 derives. What this tests is the *evidence offered for it*.

In [26]:
import exp_wagemans as WG

print(f"his 26.6 deg = arctan(1/2) = {WG.AXIS_ELEVATION:.3f}; "
      f"in our elevation set: {any(abs(e - WG.AXIS_ELEVATION) < 0.01 for e in allel)}")

print()
print("predicted measuring angles and dates:")
for nm, pairs, dd, wlat, _note in WG.SPECIMENS:
    wang = sorted(WG.measuring_angle(a, b, dd) for a, b in pairs)
    wlo, whi = 90 - wlat - WG.OBLIQUITY, 90 - wlat + WG.OBLIQUITY
    days = [WG.date_for_angle(wlat, a)[0] for a in wang if lo <= a <= hi]
    print(f"  {nm:22} {', '.join(f'{a:.1f}' for a in wang)}")
    print(f"  {'':22} -> {', '.join(WG.day_to_date(x) for x in sorted(days))}")

print()
print("1. is his deviation statistic evidence?")
dev = {}
for nm, pairs, dd, wlat, _note in WG.SPECIMENS:
    wang = [WG.measuring_angle(a, b, dd) for a, b in pairs]
    wlo, whi = 90 - wlat - WG.OBLIQUITY, 90 - wlat + WG.OBLIQUITY
    wuse = [a for a in wang if lo <= a <= hi]
    dev_obs = WG.deviation_for_angles(wlat, wuse)   # NB: not 'obs' - Part 3 binds it
    wnull = WG.null_deviation(wlat, len(wuse), trials=200)
    share = sum(1 for x in wnull if x <= dev_obs) / len(wnull)
    dev[nm] = share
    print(f"  {nm:22} observed {dev_obs:.3f} deg, random {wnull[len(wnull)//2]:.3f} deg, "
          f"{share:.0%} of random sets do as well")
print("  => the residual measures the date grid, not the object")

print()
print("2. is the sowing window a result, or forced by the geometry?")
print("  every angle must exceed 26.6 deg, and at 47-51 N the sun crosses")
print("  that band in late summer - so ANY aperture set gives that season.")
print("  (EXP-0012 confirms it with random apertures: 9 Aug to 7 Oct)")

his 26.6 deg = arctan(1/2) = 26.565; in our elevation set: True

predicted measuring angles and dates:
  Jublains (RD-0020)     41.7, 41.9, 45.1, 49.8, 50.1
                         -> 30 Aug, 30 Aug, 12 Sep, 19 Sep, 20 Sep
  Avenches (RD-0034)     38.2, 43.3, 44.2, 48.7, 49.3, 55.2
                         -> 20 Aug, 04 Sep, 06 Sep, 17 Sep, 19 Sep, 02 Oct
  Vienne (RD-0035)       47.7, 47.7, 48.5, 49.4, 51.5, 54.7
                         -> 25 Aug, 02 Sep, 08 Sep, 10 Sep, 12 Sep, 12 Sep
  Carnuntum (RD-0036)    42.9, 45.2, 48.7, 53.4, 55.3, 59.3
                         -> 16 Aug, 21 Aug, 03 Sep, 12 Sep, 17 Sep
  Tongeren (RD-0006)     33.8, 36.9, 39.1, 39.5, 40.9, 45.2
                         -> 05 Sep, 15 Sep, 19 Sep, 20 Sep, 25 Sep, 03 Oct

1. is his deviation statistic evidence?
  Jublains (RD-0020)     observed 0.115 deg, random 0.082 deg, 90% of random sets do as well
  Avenches (RD-0034)     observed 0.098 deg, random 0.081 deg, 77% of random sets do as well


  Vienne (RD-0035)       observed 0.099 deg, random 0.081 deg, 77% of random sets do as well


  Carnuntum (RD-0036)    observed 0.105 deg, random 0.082 deg, 83% of random sets do as well


  Tongeren (RD-0006)     observed 0.087 deg, random 0.081 deg, 59% of random sets do as well
  => the residual measures the date grid, not the object

2. is the sowing window a result, or forced by the geometry?
  every angle must exceed 26.6 deg, and at 47-51 N the sun crosses
  that band in late summer - so ANY aperture set gives that season.
  (EXP-0012 confirms it with random apertures: 9 Aug to 7 Oct)


## Part 9 — The blind protocols

Three protocols were run in separate sessions. The agreement rates are the
project's own reliability figures and they are the reason the results are
reported in bands rather than as a ranking.

The result files are prose tables; this part parses them and recomputes the
rates rather than quoting them.

In [27]:
def parse_ratings(path):
    # Pull (EV, direction, confidence) out of the ratings table.
    #
    # The pattern is deliberately strict - anchored on the exact four-column
    # row shape. A looser scan that hunted each row for "something that looks
    # like a direction" silently matched the wrong columns and reported 68 %
    # agreement instead of 46 %. A parser that is easy on itself is a parser
    # that agrees with whatever it is fed.
    if not os.path.exists(path):
        return None
    text = io.open(path, encoding="utf-8").read()
    rows = {}
    for m in re.finditer(
            r"^\|\s*(EV\d{3})\s*\|[^|]*\|\s*`?(\w+)`?\s*\|\s*([A-E])\s*\|",
            text, re.M):
        d = m.group(2).strip().lower()
        assert d in DIR, f"{m.group(1)}: {d!r} is not a direction"
        rows[m.group(1)] = (d, m.group(3))
    return rows

path = os.path.join(ROOT, "docs", "A3b_DIRECTION_RATINGS.md")
blind = parse_ratings(path)
if blind:
    shared = sorted(ev for ev in blind if ev in obs)
    same_d = [ev for ev in shared if blind[ev][0] == obs[ev]["direction"]]
    same_c = [ev for ev in shared if blind[ev][1] == obs[ev]["confidence"]]
    both   = [ev for ev in shared if ev in same_d and ev in same_c]
    print(f"A3b independent direction rating, recomputed from {os.path.basename(path)}")
    print(f"  rating rows parsed      {len(blind)}")
    print(f"  variables compared      {len(shared)}")
    print(f"  direction agreement     {len(same_d)}/{len(shared)} = {100*len(same_d)/len(shared):.0f} %")
    print(f"  confidence agreement    {len(same_c)}/{len(shared)} = {100*len(same_c)/len(shared):.0f} %")
    print(f"  both                    {len(both)}/{len(shared)} = {100*len(both)/len(shared):.0f} %")

    flips = [ev for ev in shared if DIR[blind[ev][0]] * DIR[obs[ev]["direction"]] < 0]
    print(f"\n  outright polarity reversals: {len(flips)}  {flips}")
    print("\n  every disagreement:")
    for ev in shared:
        if ev not in same_d:
            mark = "  <- reversal" if ev in flips else ""
            print(f"    {ev}  blind={blind[ev][0]:16} project={obs[ev]['direction']:16}{mark}")
else:
    print(f"{path} not found — the blind result files are not in this checkout")

A3b independent direction rating, recomputed from A3b_DIRECTION_RATINGS.md
  rating rows parsed      28
  variables compared      28
  direction agreement     13/28 = 46 %
  confidence agreement    15/28 = 54 %
  both                    7/28 = 25 %

  outright polarity reversals: 4  ['EV003', 'EV010', 'EV012', 'EV044']

  every disagreement:
    EV003  blind=confirmed        project=absent            <- reversal
    EV006  blind=ambiguous        project=confirmed       
    EV010  blind=confirmed        project=absent            <- reversal
    EV012  blind=weak_absent      project=weak_confirmed    <- reversal
    EV013  blind=weak_confirmed   project=confirmed       
    EV014  blind=ambiguous        project=confirmed       
    EV017  blind=weak_absent      project=absent          
    EV018  blind=weak_absent      project=absent          
    EV020  blind=ambiguous        project=absent          
    EV033  blind=weak_confirmed   project=confirmed       
    EV035  blind=ambiguous 

The A3a blind matrix is recorded as an experiment outcome. The figure that
matters is what the independently specified matrix does to the score.

In [28]:
for r in q("SELECT * FROM experiments WHERE exp_id = 'EXP-0008'"):
    print(r["outcome"][:1200])

THE ANALYSIS DEPENDS HEAVILY ON WHO SPECIFIED IT. A3a: the independent matrix agrees with the existing one on 22 of 42 comparable cells, 52 per cent, with six disagreements of two levels or more. Scored against the corpus the blind matrix gives H012 +16.4 unclustered and +17.7 clustered, against +22.7 and +22.1 for the existing matrix - a loss of about six points. A3b: direction agreement 13 of 28, 46 per cent; confidence agreement 15 of 28, 54 per cent; both agreeing on only 7 of 28, 25 per cent. Four ratings are outright polarity reversals. A5: 51 of 84 cells were respecified, so 61 per cent of the vague predictions changed once someone was asked to make them specific.


## Part 10 — What would actually change the answer

The corrections made during this project moved the reported figures but never
the baseline ranking. This part asks what *would* move it.

In [29]:
def lead(cells, clusters=None):
    t = S.totals(cells, H, clusters=clusters, tie_rule="conservative") if clusters else S.totals(cells, H)
    o = sorted(t, key=lambda h: -t[h])
    return o[0], t[o[0]] - t[o[1]]

base_leader, base_margin = lead(theirs)
print(f"baseline leader: {base_leader} by {base_margin:+.1f}\n")

print("leave one variable out:")
flips = []
for ev in sorted(disc):
    c = S.score_all(H, V, HPM, CORPUS, READINGS, include=disc - {ev})
    l, m = lead(c)
    if l != base_leader:
        flips.append((ev, l, m))
        print(f"  remove {ev} -> {l} leads by {m:+.1f}")
print(f"  {len(flips)} of {len(disc)} single variables flip the leader")

print("\nif every '-' cell justified by indifference were scored 0 (RDORP-013 A15):")
IND = re.compile(r"need not|not required|not expected|no reason to expect|does not require", re.I)
rat_col = "rationale" if "rationale" in [d[0] for d in con.execute("SELECT * FROM hpm LIMIT 1").description] else "reasoning"
h2 = dict(HPM)
n = 0
for r in q("SELECT * FROM hpm"):
    if r["prediction"] == "-" and IND.search(str(r[rat_col] or "")) and r["ev_id"] in CORPUS:
        h2[(r["hypothesis_id"], r["ev_id"])] = "0"
        n += 1
c2 = S.score_all(H, V, HPM if n == 0 else h2, CORPUS, READINGS)
l, m = lead(c2)
lc, mc = lead(c2, CLUSTERS)
print(f"  {n} cells changed -> unclustered {l} by {m:+.1f}; clustered {lc} by {mc:+.1f}")

print("\nif the two never-examined wear variables came back positive (B1):")
import dataclasses
for ev in ("EV019", "EV020"):
    c3 = dict(CORPUS)
    c3[ev] = dataclasses.replace(CORPUS[ev], direction="confirmed")
    cells3 = S.score_all(H, V, HPM, c3, READINGS)
    l, m = lead(cells3)
    print(f"  {ev} confirmed -> {l} leads by {m:+.1f}")
print("\n  B1 does not decide the leader: the wear cluster scores identically for H012 and H014.")

baseline leader: H012 by +3.0

leave one variable out:
  remove EV039 -> H014 leads by +0.6
  1 of 32 single variables flip the leader

if every '-' cell justified by indifference were scored 0 (RDORP-013 A15):
  13 cells changed -> unclustered H014 by +1.0; clustered H014 by +1.1

if the two never-examined wear variables came back positive (B1):
  EV019 confirmed -> H012 leads by +3.0
  EV020 confirmed -> H012 leads by +3.0

  B1 does not decide the leader: the wear cluster scores identically for H012 and H014.


## Part 11 — All results in one place

One cell, everything the project concludes. Nothing below is retyped: every
figure comes from the objects computed above, and the results table is the one
`render_docs` writes into RDORP-012.

In [30]:
line = lambda c="-": print(c * 78)

line("=")
print("RDORP — CONSOLIDATED RESULTS".center(78))
line("=")
print(f"database sha256 {digest[:16]}...   tie rule {RD.TIE_RULE!r}")

line()
print("CORPUS")
line()
print(f"  {specimens} specimens, {observations} sourced observations, {sources} sources, "
      f"{countries} countries")
print(f"  coverage {100*specimens/KNOWN_CORPUS:.0f} % of {KNOWN_CORPUS} catalogued; "
      f"{100*british/specimens:.0f} % British against a known corpus about 20 % British")
print(f"  {fragments} fragments; {len(disc)} of {q1('SELECT COUNT(*) FROM evidence_variables')} "
      f"evidence variables scored")
print(f"  {100*top_src['c']/observations:.0f} % of observations come from "
      f"{top_src['source_id']} alone; two sources account for {top2_pct} %")
print(f"  admissible: mass {admit['Mass']}, geometry {admit['Geometry']}, "
      f"context {admit['Context']} of {specimens}")

line()
print("RANKING  (bands are a judgement; scores are not)")
line()
band_of = {h: lbl.strip('*') for lbl, ms in RD.BANDS for h in ms}
print(f"  {'H':5} {'clustered':>10} {'unclustered':>12} {'staked':>7} {'ratio':>6} "
      f"{'value':>6}  {'band':<30} name")
for lbl, members in RD.BANDS:
    for h in members:
        mp = com[h]["max_possible"]
        print(f"  {h:5} {C[h]:+10.1f} {U[h]:+12.1f} {mp:7.0f} {U[h]/mp:6.2f} "
              f"{facts.value.get(h,0):+6} {band_of[h]:<30} {names[h][:34]}")

line()
print("STABILITY")
line()
print(f"  leader, baseline                {base_leader} by {base_margin:+.1f}")
print(f"  leaders across {len(scenarios)} scenarios       {sorted(leaders)}")
for label, w in (("unclustered", sweep_u), ("clustered", sweep_c)):
    for h, n in sorted(w["leaders"].items(), key=lambda x: -x[1]):
        print(f"  weight sweep, {label:12}   {h} leads {n}/{w['n']}  "
              f"margins {w['min_margin']:+.1f} to {w['max_margin']:+.1f}")
print(f"  single variables that flip it   {[e for e, _l, _m in flips] or 'none'}")
print(f"  largest single weighted cell    {max(abs(v[4]) for v in theirs.values()):.1f}"
      f"   (vs a leader margin of {base_margin:.1f})")

line()
print("RELIABILITY  — the reason results are banded, not ranked")
line()
if blind:
    print(f"  independent direction rating    {len(same_d)}/{len(shared)} "
          f"= {100*len(same_d)/len(shared):.0f} % agreement")
    print(f"  independent confidence grading  {len(same_c)}/{len(shared)} "
          f"= {100*len(same_c)/len(shared):.0f} %")
    print(f"  both                            {len(both)}/{len(shared)} "
          f"= {100*len(both)/len(shared):.0f} %")
print("  independent prediction matrix   22/42 = 52 % of cells (EXP-0008), "
      "and it scored H012 six points lower")

line()
print("COMPUTATIONAL RESULTS  — independent of every judgement above")
line()
print(f"  vertex-transitive: all {len(verts)} knobs identical      -> knob choice conveys nothing")
print(f"  face axes take only {len(angs)} distinct angles {angs}")
print(f"     smallest is {min(angs):.1f}deg vs a {ANNUAL_SWING:.1f}deg annual solar swing "
      f"-> cannot index 12 dates")
print(f"  suspension gives {len(allel)} distinct elevations, 4 reachable -> 8 events, not 12")
print(f"  ring limit cos36 = {RATIO:.5f} -> no ring beyond 80.9 % of the knob radius")
print(f"  ring counts span 0-6, so >= {collisions} of 12 faces must collide -> cannot label 12 signs")
print(f"  rotation group order {len(group)}; marking one axis leaves {len(stab)} "
      f"-> an axis is not an orientation")
print(f"  zodiac fit {best:.2f}deg vs {median_random:.2f}deg for the median random "
      f"solid; {p_zod:.0%} of random sets fit as well -> the alignment is the scan")
print(f"  internal volume {min(vols):.0f}-{max(vols):.0f} ml; fit to Roman units no "
      f"better than chance ({p_vol:.0%}); retained volume 0 ml -> not a measure")
print(f"  smallest aperture step {min(gaps.values()):.2f}mm -> the object cannot be "
      f"READ as a gauge; but only {n_twelve} specimen has all twelve measured")
print(f"  best aperture pair levels to {sight_tolerance(14.2,14.5,46.5):.2f}deg "
      f"-> 10x too coarse for Nimes")

line()
print("KNOWN DEFECTS IN THE ANALYSIS ITSELF")
line()
nosrc = q1("SELECT COUNT(*) FROM corpus_observations "
           "WHERE source_id IS NULL OR source_id = ''")
print(f"  unwritten predictions on scored variables   {len(unwritten)} cells (A18)")
print(f"  scored variables with no source at all      {nosrc} (A4)")
print(f"  opposite-sign ties inside a cluster         {ties} (A16, fixed)")
print(f"  the leader turns on                         EV039 alone")
line("=")

                         RDORP — CONSOLIDATED RESULTS                         
database sha256 8f682c1f023818f2...   tie rule 'conservative'
------------------------------------------------------------------------------
CORPUS
------------------------------------------------------------------------------
  60 specimens, 238 sourced observations, 51 sources, 10 countries
  coverage 45 % of 134 catalogued; 38 % British against a known corpus about 20 % British
  11 fragments; 32 of 48 evidence variables scored
  38 % of observations come from PUB-0006 alone; two sources account for 53 %
  admissible: mass 6, geometry 11, context 50 of 60
------------------------------------------------------------------------------
RANKING  (bands are a judgement; scores are not)
------------------------------------------------------------------------------
  H      clustered  unclustered  staked  ratio  value  band                           name
  H012       +23.5        +24.0      36   0.67     -1 Lead

## Part 12 — Assertions

Every headline figure the documents publish, checked against what this notebook
computed. **If the corpus changes and a document is not regenerated, this cell
fails.**

In [31]:
FAILURES = []
def expect(label, got, want, tol=None):
    # A tolerance is given explicitly where a figure is sampled rather than
    # exact - the Monte Carlo percentage moves by a point with the trial count.
    if tol is None:
        tol = 0.05 if isinstance(want, float) else 0
    ok = abs(got - want) <= tol if isinstance(want, (int, float)) else got == want
    print(f"  {'OK  ' if ok else 'FAIL'}  {label:52} got {got}, expect {want}")
    if not ok:
        FAILURES.append(label)

expect("specimens",                       specimens, 60)
expect("sourced observations",            observations, 238)
expect("sources",                         sources, 51)
expect("countries",                       countries, 10)
expect("evidence variables",              q1("SELECT COUNT(*) FROM evidence_variables"), 48)
expect("hypotheses",                      len(hyps), 14)
expect("experiments",                     q1("SELECT COUNT(*) FROM experiments"), 12)
expect("pre-registered predictions",      q1("SELECT COUNT(*) FROM predictions"), 11)
expect("screened domains",                q1("SELECT COUNT(*) FROM screening_candidates"), 17)
expect("scored variables",                len(disc), 32)
expect("British share (%)",               round(100*british/specimens), 38)
expect("coverage (%)",                    round(100*specimens/KNOWN_CORPUS), 45)

expect("baseline leader",                 base_leader, "H012")
expect("H012 unclustered",                round(U["H012"], 1), 24.0)
expect("H014 unclustered",                round(U["H014"], 1), 21.0)
expect("H012 clustered",                  round(C["H012"], 1), 23.5)
expect("H014 clustered",                  round(C["H014"], 1), 20.5)
expect("H013 clustered (largest riser)",  round(C["H013"], 1), 17.1)
expect("H009 unclustered (eliminated)",   round(U["H009"], 1), -34.0)

expect("rotation group order",            len(group), 60)
expect("stabiliser of one marked axis",   len(stab), 10)
expect("vertices",                        len(verts), 20)
expect("distinct face-axis angles",       len(angs), 3)
expect("cos 36 deg",                      round(RATIO, 5), 0.80902)
expect("distinct suspension elevations",  len(allel), 7)
expect("zodiac best fit (deg)",           round(best, 2), 4.98)
expect("zodiac best-fit latitude",        round(lat, 1), 58.0)
expect("zodiac chance baseline (deg)",    Z.EXPECTED_AT_RANDOM, 7.5)
expect("random sets fitting as well (%)", round(100 * p_zod), 84, tol=2)
expect("volume coefficient",              round(EV.VOL_COEFF, 4), 7.6631)
expect("largest internal volume (ml)",    round(max(vols)), 1298, tol=1)
expect("volumes as close to a unit as chance (%)", round(100 * p_vol), 65, tol=2)
expect("specimens with all 12 apertures", n_twelve, 2)
expect("Avenches beats this share of random sets (%)",
       round(100 * max(graded.values())), 83, tol=3)

if blind:
    expect("A3b direction agreement",         len(same_d), 13)
    expect("A3b confidence agreement",        len(same_c), 15)
    expect("A3b both",                        len(both), 7)
    expect("A3b variables compared",          len(shared), 28)

expect("provenance A",                    grades.get("A", 0), 1)
expect("provenance C",                    grades.get("C", 0), 27)
expect("provenance E",                    grades.get("E", 0), 4)
expect("provenance D",                    grades.get("D", 0), 26)
expect("admissible for mass",             admit["Mass"], 6)
expect("admissible for geometry",         admit["Geometry"], 11)
expect("admissible for context",          admit["Context"], 50)
expect("fragments",                       fragments, 11)
expect("top-source share (%)",            round(100*top_src["c"]/observations), 38)
expect("top-two-source share (%)",        top2_pct, 53)

print()
if FAILURES:
    raise AssertionError(f"{len(FAILURES)} assertion(s) failed: {FAILURES}")
print("All assertions passed - the documents match the database.")

  OK    specimens                                            got 60, expect 60
  OK    sourced observations                                 got 238, expect 238
  OK    sources                                              got 51, expect 51
  OK    countries                                            got 10, expect 10


  OK    evidence variables                                   got 48, expect 48
  OK    hypotheses                                           got 14, expect 14
  OK    experiments                                          got 12, expect 12
  OK    pre-registered predictions                           got 11, expect 11
  OK    screened domains                                     got 17, expect 17
  OK    scored variables                                     got 32, expect 32
  OK    British share (%)                                    got 38, expect 38
  OK    coverage (%)                                         got 45, expect 45
  OK    baseline leader                                      got H012, expect H012
  OK    H012 unclustered                                     got 24.0, expect 24.0
  OK    H014 unclustered                                     got 21.0, expect 21.0
  OK    H012 clustered                                       got 23.5, expect 23.5
  OK    H014 clustered              

---

## What this notebook does not settle

It shows the numbers follow from the data by the stated rules. It says nothing
about whether the data is right.

- Two independent specifiers agreed on **52 %** of one hypothesis's prediction
  cells; two raters agreed on **46 %** of directions (Part 9).
- Four scored variables carry **no predictions at all**, and those cells score
  zero by default rather than by judgement (Part 3).
- The leader turns on **one variable**, EV039 (Part 10), and on how a handful of
  "not required" cells are scored — both still open in RDORP-013.

Reproducibility is a floor, not a result.